# The Function Guessing Game

In [ ]:
#@title Verification code

import itertools
import time
import numpy as np
import warnings
import random
import re
import sympy


def random_sympy_poly_hidden(max_degree=5, max_coeff=5):
  """Generates a random sympy polynomial."""
  x = sympy.symbols('x')
  degree = random.randint(1, max_degree)
  coeffs = [random.randint(-max_coeff, max_coeff) for _ in range(degree + 1)]
  poly = sum(c * x**i for i, c in enumerate(coeffs))
  return poly


def generate_functions():
  """Generates a list of diverse sympy functions, ordered by difficulty."""
  x = sympy.symbols('x')
  functions = []

  # Tier 1: Basic and simple polynomials
  functions.append(x * random.randint(-5, 5))
  functions.append(x * random.randint(-5, 5) + random.randint(-5, 5))
  functions.append(x**2 + random.randint(-5, 5))
  for _ in range(3):
    functions.append(random_sympy_poly_hidden(max_degree=3, max_coeff=5))

  # Tier 2: Trigonometric and exponential
  functions.append(sympy.sin(x) + random.randint(-5, 5))
  functions.append(sympy.cos(x) + random.randint(-5, 5))
  functions.append(sympy.exp(x) + random.randint(-5, 5) * x)

  # Tier 3: Combinations
  functions.append(3 * sympy.sin(x) * sympy.cos(x) + random.randint(-5, 5))
  functions.append(sympy.log(x + random.randint(5, 10)))

  return functions


def evaluate_guess(guess, function, test_points=None):
  """Evaluates how close a guess is to the actual function."""
  if test_points is None:
    test_points = np.linspace(-10, 10, 100)

  x = sympy.symbols('x')
  try:
    f_actual = sympy.lambdify(x, function, 'numpy', cse=True)
    f_guess = sympy.lambdify(x, guess, 'numpy', cse=True)
    with warnings.catch_warnings():
      warnings.simplefilter('ignore')
      l1_error = np.nanmean(np.abs(f_actual(test_points) - f_guess(test_points)))
    return l1_error
  except Exception:
    return float('inf')

In [ ]:
#@title Initial program

import time
import sympy
import logging


def function_guessing_strategy():
  """Strategy to guess a hidden function.

  This problem uses LLM calls (call_llm) and oracle queries
  (send_question_to_oracle, ask_for_function_value_at) to iteratively
  narrow down the form of a hidden mathematical function.

  The strategy:
  1. Ask yes/no questions to the oracle to determine the function's form
  2. Evaluate the function at several test points
  3. Use an LLM to fit coefficients to the identified function form

  Returns:
      A sympy expression representing the guessed function.
  """
  start_time = time.time()
  llm_call_count = 0

  first_llm_call = 'Is the first part of the hidden function a polynomial?'
  try:
    answer = send_question_to_oracle(first_llm_call)
    llm_call_count += 1
  except Exception:
    answer = "I don't know"

  all_conversation = first_llm_call + '\n' + answer + '\n'

  while time.time() - start_time < 970 and llm_call_count < 50:
    prompt = (
        "I am trying to guess a secret function. Conversation so far:\n"
        + all_conversation
        + "\nWhat yes/no question should I ask next? Reply only with the question."
    )
    try:
      next_question = call_llm(prompt)
      llm_call_count += 1
    except Exception:
      next_question = 'Is it a trigonometric function?'

    try:
      answer = send_question_to_oracle(next_question)
      llm_call_count += 1
    except Exception:
      answer = "I don't know"

    all_conversation += next_question + '\n' + answer + '\n'

  # Evaluate at test points
  values = {}
  for x_val in [0, 1, 2.1, 3, 4, 4.3, 0.3, 0.5, 0.7, -4.3]:
    values[x_val] = ask_for_function_value_at(x_val)

  # Ask LLM to determine the function form and fit coefficients
  prompt = (
      "Based on the following conversation, guess the form of the hidden function:\n"
      + all_conversation
  )
  try:
    final_function_format = call_llm(prompt)
    llm_call_count += 1
  except Exception:
    final_function_format = "unknown"

  # Generate code to find coefficients
  prompt = (
      "Return a Python function called `find_constants_in_hidden_function` "
      "that takes a dictionary of values at different x-values and returns "
      "the coefficients as a sympy expression.\n"
      f"Values: {values}\n"
      f"Function form: {final_function_format}"
  )
  try:
    code = call_llm(prompt)
    exec(code, globals())
    function_guess = find_constants_in_hidden_function(values)
  except Exception:
    function_guess = None

  return function_guess

**Prompt used**

Act as an expert mathematician and software developer specializing in symbolic computation and function identification.

Your task is to write a Python function, function_guessing_strategy, which attempts to discover a hidden mathematical function. This function will be evaluated in a game-like environment where it must intelligently query an oracle and sample function points to make a final guess.

The evaluation function, which you do not need to write, scores your strategy based on the following rules:

Objective: Correctly guess the hidden function, which is a sympy expression.

Tools: Your function_guessing_strategy can call three special functions:

send_question_to_oracle(prompt: str) -> str: You can ask high-level, natural language questions about the function's properties (e.g., "Is the function periodic?"). The oracle will only ever respond with the string "Yes" or "No". You are strictly limited to a maximum of 100 calls to this function. Exceeding this limit results in a massive penalty.

call_llm(prompt: str) -> str: You can call an independent LLM directly with this function. This LLM doesn't know anything about the hidden function, but can answer any question, not just yes/no questions. You can talk to it like to any regular LLM. You can also ask it to give you custom code (for example to find the exact coefficients and constants in the hidden function, once you have guessed its overall form by asking yes/no questions to the oracle), and you can run this code instantly. Every call to this LLM counts towards the global limit of maximum 100 LLM calls.

ask_for_function_value_at(x: float) -> float: You can get the specific value of the hidden function at any point x. You can call this as many times as you like, but each call incurs a small penalty. Keep in mind that some of the functions might have singularities at some points, in which case this function might return np.inf or np.nan values.

Scoring:

A correct guess earns a high base score.

A penalty of 10 points is subtracted for each call to ask_for_function_value_at.

A penalty of 1 point is subtracted for each call to call_llm and send_question_to_oracle.

An incorrect guess or returning None results in a significant penalty.

Exceeding the 100-question limit results in disqualification for that round.

Time Limit: Your strategy will have approximately 1000 seconds to run for each hidden function. After this time, it must return its best guess.

Your goal is to write a function_guessing_strategy that maximizes this score across a wide variety of hidden functions (including polynomials, trigonometric, exponential, discontinuous functions, and linear / non-linear combinations of such functions).

A robust strategy might first use the oracle to determine the general family of the function. You are also allowed to ask the LLM to give you the sympy expression for the final function guess, based on all the natural language answers given by the oracle -- just note that this also counts as a question, so it counts towards your 100 question limit. You can query an LLM directly with the call_llm(prompt) function.

You must return a valid sympy expression as your final guess. You will see which functions the previous program got wrong in the comments after the import statements of the previous code, labeled with lines such as "f_27_iqhd = ...". You will receive different functions to guess each time, so there is no point in memorising these ad verbatim, but you should always focus on being able to guess the functions you weren't able last time.

Before you return your guess, make sure you log all the conversation you've had with the LLM, using logging.info() commands.

## What AlphaEvolve found

This experiment tested AlphaEvolve's ability to write code that itself makes LLM calls. AlphaEvolve evolved programs that would ask an oracle simple yes/no questions about the hidden function (e.g., "Is the function periodic?", "Is the function a polynomial?"), collect the answers, and then make a final LLM call asking for custom search code to identify the exact function form and coefficients. While AlphaEvolve managed to outperform baseline algorithms that were not allowed to make LLM calls, the approach had practical limitations: the cheap oracle LLM gave noisy answers for complex functions, and initially even leaked the hidden function's identity when questions were phrased in certain ways. The non-oracle LLM was also not always reliable at returning good search code.